# AGENTES LÓGICOS COM BASES DE CONHECIMENTO

## Bases de conhecimento proposicionais: `PropKB`

A classe `PropKB` pode ser utilizada para representar uma base de conhecimento de frases lógicas proposicionais:

* `__init__(self, sentence=None)`: O construtor `__init__` cria um único campo `clauses` que será uma lista de todas as frases da base de conhecimento. Note-se que cada uma destas frases será uma 'cláusula', ou seja, uma frase composta apenas por literais e `ou`'s.
* `tell(self, sentence)`: Quando pretende adicionar uma frase à KB, utilize o método `tell`. Este método recebe uma frase, converte-a no seu CNF, extrai todas as cláusulas e adiciona todas essas cláusulas ao campo `clauses`. Portanto, não tem de se preocupar em `tell` apenas cláusulas para a base de conhecimento. Pode `tell` à base de conhecimento uma frase em qualquer formato que desejar; convertê-lo para CNF e adicionar as cláusulas resultantes será feito pelo método `tell`.
* `ask_generator(self, query)`: A função `ask_generator` é utilizada pela função `ask`. Chama a função `tt_entails`, que por sua vez retorna `True` se a base de conhecimento implicar a consulta e `False` caso contrário. O próprio `ask_generator` devolve um dicionário vazio `{}` se a base de conhecimento implicar a consulta e `None` caso contrário. Isto pode parecer-lhe um pouco estranho. Afinal, faz mais sentido retornar apenas `True` ou `False` em vez de `{}` ou `None`. Mas isto é feito para manter a consistência com a forma como as coisas são na Lógica de Primeira Ordem, onde uma função `ask_generator` deve devolver todas as substituições que tornam a consulta verdadeira. Daí o `dict`, para devolver todas estas substituições. Voçê vai utilizar principalmente a função `ask`, que retorna `{}` ou `False`, mas se não gostar, pode sempre utilizar a função `ask_if_true`, que retorna `True` ou `False`.
* `retract(self, sentence)`: Esta função remove todas as cláusulas da frase fornecida da base de conhecimento. Tal como a função `tell`, não precisa de passar cláusulas para as remover da base de conhecimento; qualquer frase serve. A função encarregar-se-á de converter essa frase em cláusulas e depois removê-las.

## Agentes baseados em conhecimento

Um agente baseado em conhecimento é um agente genérico simples que mantém e gere uma base de conhecimento.
A base de conhecimento pode inicialmente conter algum conhecimento prévio.
<br>
O objetivo de um agente KB é fornecer um nível de abstração sobre a manipulação da base de conhecimento e deve ser utilizado como uma classe base para os agentes que trabalham numa base de conhecimento.
<br>
Dado um percepto, o agente KB adiciona o percepto à sua base de conhecimento, pergunta à base de conhecimento qual é a melhor ação e informa a base de conhecimento que de facto executou essa ação.
<br>
A nossa implementação de `KB-Agent` está encapsulada numa classe `KB_AgentProgram` que herda da classe `KB`.
<br>
Vamos dar uma vista de olhos.

In [ ]:
psource(KB_AgentProgram)

As funções auxiliares `make_percept_sentence`, `make_action_query` e `make_action_sentence` são todas apropriadamente nomeadas e, como esperado,
`make_percept_sentence` cria frases lógicas de primeira ordem sobre percepções que queremos que o nosso agente receba,
`make_action_query` pergunta ao `KB` subjacente sobre a ação que deve ser tomada e
`make_action_sentence` informa o `KB` subjacente sobre a ação que acabou de executar.

## MUNDO WUMPUS

O mundo dos wumpus é uma gruta composta por salas ligadas por passagens. Algures na gruta está o terrível `wumpus`, uma besta que devora qualquer pessoa que entre no seu quarto. O wumpus pode ser atingido por um agente, mas o agente tem apenas uma flecha. Algumas salas contêm poços sem fundo que prendem qualquer pessoa que entre nelas (excepto os wumpus, que são demasiado grandes para cair). A única característica redentora deste ambiente sombrio é a possibilidade de encontrar uma pilha de ouro.

Embora o mundo dos wumpus seja bastante inofensivo para os padrões dos jogos de computador modernos, ilustra alguns pontos importantes sobre a inteligência.

<img src="./images/wumpus.png">

Um exemplo de um mundo wumpus é apresentado na figura acima. A definição precisa do ambiente da tarefa é dada pela descrição PEAS:

- **Medida de desempenho**: +1000 para sair da caverna com o ouro, –1000 para cair num poço ou ser comido pelo wumpus, –1 por cada acção realizada e –10 para utilizar a flecha. O jogo termina quando o agente morre ou quando sai da caverna.

- **Ambiente**: Uma grade 4X4 de divisões, com paredes a envolver a grade. O agente começa sempre no quadrado rotulado [1,1], virado para Este. Os locais do ouro e do wumpus são escolhidos aleatoriamente, com uma distribuição uniforme, entre os quadrados diferentes do quadrado inicial. Além disso, cada quadrado diferente do inicial pode ser um buraco, com uma probabilidade de 0,2.

- **Atuadores**: O agente pode deslocar-se para a frente, virar à esquerda a 90º ou virar à direita a 90º. O agente sofre uma morte miserável se entrar num quadrado contendo um poço ou um wumpus vivo. (É seguro, embora malcheiroso, entrar num quadrado com um wumpus morto.) Se um agente tentar avançar e bater numa parede, o agente não se move. A ação Agarrar pode ser utilizada para apanhar o ouro se este estiver no mesmo quadrado que o agente. A ação Disparar pode ser utilizada para disparar uma flecha em linha reta na direção para onde o agente está a olhar. A flecha continua até atingir (e, portanto, matar) o wumpus ou atingir uma parede. O agente tem apenas uma flecha, pelo que apenas a primeira ação Disparar tem algum efeito. Por fim, a ação Escalar pode ser utilizada para sair da gruta, mas apenas a partir do quadrado [1,1].

- **Sensores**: O agente possui cinco sensores, cada um dos quais fornece um único bit de informação:
    
    – Nos quadrados directamente (não diagonalmente) adjacentes ao wumpus, o agente irá percepcionar um Fedor(`Stench`).
    
    – Nos quadrados directamente adjacentes a um poço, o agente irá percepcionar uma Brisa (`Breeze`).
    
    – No quadrado onde está o ouro, o agente irá reparar num Brilho (`Glitter`).
    
    – Quando um agente embate numa parede, repara num Encontrão (`Bump`).
    
    – Quando o wumpus é morto, emite um grito lamentável que pode ser percebido em qualquer parte da gruta.
    
As perceções serão fornecidas ao programa agente sob a forma de uma lista de cinco símbolos; por exemplo, se houver um fedor e uma brisa, mas nenhum brilho, solavanco ou grito, o programa do agente obterá [*Stench;Breeze;None;None;None*].

Podemos caracterizar o ambiente `wumpus` ao longo das várias dimensões:

 - É um mundo claramente determinístico, discreto, estático e de agente único. 
 
 - É não episódico porque as recompensas podem vir apenas depois de muitas ações terem sido tomadas.
 
 - É parcialmente observável, porque alguns aspetos do estado não são diretamente percetíveis: a localização do agente, o estado de saúde do wumpus e a disponibilidade de uma flecha.
 
 Quanto às **localizações dos poços e dos wumpus**: poderíamos tratá-los como partes não observadas do estado — nesse caso, o modelo de transição para o ambiente é completamente conhecido, e encontrar as localizações dos poços completa o conhecimento do agente sobre o estado.
 
 Em alternativa, poderíamos dizer que o modelo de transição em si é desconhecido porque o agente não sabe quais as ações *Forward* que são fatais — neste caso, descobrir as **localizações dos pits e do wumpus** completa o conhecimento do agente sobre o modelo de transição.

In [ ]:
from utils import *
from logic4e import *
from search import breadth_first_graph_search
from notebook import psource

## Base de conhecimento do mundo Wumpus

Vamos criar uma base de conhecimento para o mundo `wumpus`. Vamos usar a classe `PropKB` e aplicar as regras do mundo.

In [ ]:
wumpus_kb = PropKB()

Definimos os simbolos que vamos usar nas nossas cláusulas:<br/><br/>
$P_{x,y}$ é verdadeira se existir um poço em [x, y].<br/>
$W_{x,y}$ é verdadeira se existir um wumpus em [x, y], vivo ou morto.<br/>
$B_{x,y}$ é verdadeira se houver uma brisa em [x, y].<br/>
$S_{x,y}$ é verdadeira se existir um fedor em [x, y].<br/>
$L_{x,y}$ é verdadeira se o agente estiver no local [x, y].<br/>

Vamos para já ter em atenção só as células acessiveis no inicio do jogo em [1, 1] e [1, 2]

In [ ]:
P11, P12, P21, P22, P31, B11, B21 = expr('P11, P12, P21, P22, P31, B11, B21')

Agora dizemos as proposições (regras) $R_i$ à nossa base de conhecimento.<br/><br/>

Não há nenhum poço em `[1,1]`:

$R_1: \neg P_{1,1}$


In [ ]:
wumpus_kb.tell(~P11)

Um quarto é arejado se, e só se, existir um poço num quarto vizinho. Isto precisa de ser declarado para cada quarto, mas, por enquanto, apenas incluímos os quartos relevantes:

$R_2: B_{1,1} \iff (P_{1,2} \lor P_{2,1} )$

$R_3: B_{2,1} \iff (P_{1,1} \lor P_{2,2} \lor P_{3,1} )$

In [ ]:
wumpus_kb.tell(B11 | '<=>' | ((P12 | P21)))
wumpus_kb.tell(B21 | '<=>' | ((P11 | P22 | P31)))

Incluímos agora as percepções da brisa para os dois primeiros quartos:

$R_4: \neg B_{1,1}$

$R_5: B_{2,1}$

In [ ]:
wumpus_kb.tell(~B11)
wumpus_kb.tell(B21)

Podemos verificar as cláusulas guardadas num `KB` acedendo à sua variável `clauses`

In [ ]:
wumpus_kb.clauses

Vemos que a equivalência $B_{1, 1} \iff (P_{1, 2} \lor P_{2, 1})$ foi automaticamente convertida em duas implicações que por sua vez foram convertidas em CNF que é armazenado no `KB`. <br/><br/>
$B_{1, 1} \iff (P_{1, 2} \lor P_{2, 1})$ foi dividido em $B_{1, 1} \implies (P_{1, 2} \lor P_{2, 1})$ e $B_{1, 1} \Longleftarrow (P_{1, 2} \lor P_{2, 1})$.<br/>
$B_{1, 1} \implies (P_{1, 2} \lor P_{2, 1})$ foi convertido para $P_{1, 2} \lor P_{2, 1} \lor \neg B_{1, 1}$.<br/>
$B_{1, 1} \Longleftarrow (P_{1, 2} \lor P_{2, 1})$ foi convertido para $\neg (P_{1, 2} \lor P_{2, 1}) \lor B_{1, 1}$ que se transforma em $(\neg P_{1, 2} \lor B_{1, 1}) \land (\neg P_{2, 1} \lor B_{1, 1 })$ após a aplicação das leis de De Morgan e distribuição da disjunção. <br/>
$B_{2, 1} \iff (P_{1, 1} \lor P_{2, 2} \lor P_{3, 2})$ é convertido de forma semelhante.




## Inferência

O nosso objectivo é agora decidir se $KB \models \alpha$ para alguma proposição $\alpha$, por exemplo, se a proposição $\alpha = \neg P_{1,2}$ está implicado na nossa base de conhecimento $KB$?

## Inferência por teste de modelos

Um primeiro algoritmo para inferência é uma abordagem de verificação de modelos que é uma implementação direta da definição de implicação: enumerar os modelos e verificar se isto é verdade em todos os modelos em que $KB$ é verdadeiro.

<img src="./images/modelos.png">

Uma tabela de verdade construída para a base de conhecimento fornecido. A tabela de verdade tem $2^7=128$ linhas. $KB$ é verdadeira se $R_1$ a $R_5$ forem verdadeiras, o que acontece em apenas 3 das 128 linhas (as sublinhadas na coluna da direita). Em todas as 3 linhas, $P_{1,2}$ é falso, logo não existe poço em [1,2]. Por outro lado, pode (ou não) existir um poço em [2,2].

Se $KB$ e $\alpha$ contiverem $n$ símbolos no total, então existem $2^n$ modelos. Assim, a complexidade temporal do algoritmo é $O(2^n)$ (A complexidade do espaço é apenas $O(n)$ porque a enumeração é em profundidade.) 

Infelizmente, a implicação proposicional é co-NP-completa (i.e., provavelmente não é mais fácil do que NP-completa), pelo que todo o algoritmo de inferência conhecido para lógica proposicional tem uma complexidade no pior caso que é exponencial em relação ao tamanho da entrada.

## Inferência por resolução lógica

Os procedimentos de inferência baseados na resolução funcionam utilizando o princípio da prova por contradição.

Ou seja, para mostrar que $ KB\models\alpha $ , mostramos que $(KB\land\neg\alpha)$ é insatisfatório. A prova é feita através da contradição.

Um algoritmo de resolução é o seguinte:

<img src="./images/pl_resolution.png">

- Em primeiro lugar, $(KB\land\neg\alpha)$ é convertido em CNF (**Forma Normal Conjuntiva**).

- De seguida, a regra de resolução é aplicada às cláusulas resultantes. Cada par que contém literais complementares é resolvido para produzir uma nova cláusula, que é adicionada ao conjunto se não estiver já presente.

- O processo continua até que uma de duas coisas aconteça:

    - não existam novas cláusulas que possam ser acrescentadas, caso em que $KB$ não implica $\alpha$ ;

    - ou duas cláusulas resolvem produzir a cláusula vazia, caso em que $KB$ implica $\alpha$.

A frase vazia — uma disjunção sem disjunções — é equivalente a *False* porque uma disjunção é verdadeira apenas se pelo menos uma das suas disjunções for verdadeira. Além disso, a proposição vazia surge apenas da resolução de duas orações unitárias contraditórias, como $P$ e $ \neg P$.

Podemos aplicar o procedimento de resolução a uma inferência muito simples no mundo wumpus. Quando o agente está em [1,1], não há brisa, pelo que não pode haver poços em quadrados vizinhos.

A base de conhecimento relevante é $KB = R_2 \land R_4 = (B_{1,1} \implies (P_{1,2} \lor P_{2,1})) \land \neg B_{1,1}$ e queremos provar $\alpha$, que é, digamos, $\neg P_{1,2}$.

Quando convertemos $(KB \land \neg \alpha )$ em CNF, obtemos as cláusulas apresentadas no topo da seuinte figura:

<img src="./images/clauses.png">

A segunda linha da figura mostra proposições obtidas por resolução de pares na primeira linha. Assim, quando $P_{1,2}$ é resolvido com $\neg P_{1,2}$, obtemos a cláusula vazia, apresentada como um pequeno quadrado.

A inspeção da figura revela que muitos passos de resolução são inúteis. Por exemplo, a cláusula $ \neg B_{1,1} \lor P_{1,2} \lor B_{1,1} $ é equivalente a $True \lor P_{1,2}$ que é equivalente a $True$. Deduzir que $True$ é verdadeiro não ajuda muito. Assim sendo, qualquer cláusula na qual apareçam dois literais complementares pode ser descartada.

In [ ]:

''' um algoritmo para resolver proposições contra uma base de conhecimento'''

def pl_resolution(kb, alpha):
    """
    Resolução de lógica proposicional
    Diz se A segue da base de conhecimentos KB:
    >>> pl_resolution(KB, A)
    True
    """
    clauses = kb.clauses + conjuncts(to_cnf(~alpha))
    new = set()
    
    while True:
        n = len(clauses)
        pairs = [(clauses[i], clauses[j])
                 for i in range(n) for j in range(i + 1, n)]
        for (ci, cj) in pairs:
            resolvents = pl_resolve(ci, cj)
            if False in resolvents:
                return True
            new = new.union(set(resolvents))
        if new.issubset(set(clauses)):
            return False
        for c in new:
            if c not in clauses:
                clauses.append(c)


def pl_resolve(ci, cj):
    """Retorna todas as clausulas que podem ser obtidas pela resolução de ci e cj."""
    clauses = []
    for di in disjuncts(ci):
        for dj in disjuncts(cj):
            if di == ~dj or ~di == dj:
                clauses.append(associate('|', unique(remove_all(di, disjuncts(ci)) + remove_all(dj, disjuncts(cj)))))
    return clauses

As funções `disjuncts`e `associate` funcionam do seguinte modo:

In [ ]:
print(disjuncts((~P | Q)))
print(associate('|',[~P,Q]))
print(associate('|',[]))

Podemos agora fazer a nossa *query*:

In [ ]:
pl_resolution(wumpus_kb, ~P12)

Outra forma de resolver uma proposição lógica é verificar a satisfatibilidade, o problema SAT.

O problema SAT (do inglês, Satisfiability Problem) consiste em determinar se, dada uma fórmula booleana, existe alguma atribuição de valores (verdadeiro ou falso) às variáveis que torne a fórmula verdadeira. Em termos simples, é o desafio de "satisfazer" uma fórmula lógica, encontrando um conjunto de valores que cumpra todas as condições impostas pela fórmula.

### Características principais do problema SAT:
- **Fórmula booleana:** A fórmula é composta por variáveis, operadores lógicos (como E, OU e NÃO) e, normalmente, apresentada numa forma normal conjuntiva (CNF).  
- **Atribuição de valores:** A tarefa consiste em verificar se existe alguma combinação dos valores atribuídos às variáveis que faça com que toda a fórmula se avalie como verdadeira.  
- **Complexidade:** O problema SAT é notório na teoria da computação porque foi o primeiro problema a ser demonstrado como NP-completo. Isto significa que, se conseguirmos encontrar um algoritmo eficiente (em tempo polinomial) para o SAT, então todos os problemas na classe NP poderão ser resolvidos eficientemente.  
- **Aplicações:** O SAT tem inúmeras aplicações práticas, desde a verificação de circuitos e a inteligência artificial até à resolução de problemas de planeamento e otimização.

O problema SAT representa um desafio central na área da computação teórica e prática, pois envolve encontrar uma forma de satisfazer uma dada expressão lógica, um problema que, apesar de simples de enunciar, esconde uma complexidade computacional profunda.

 O teste da implicação, $\beta \models \alpha$ pode ser feito testando a insatisfatibilidade de $\beta \land \neg \alpha$

In [ ]:
# outra forma de fazer resolução com satisfabilidade

def pl_resolution_2(kb, alpha):
    clauses = kb.clauses + conjuncts(to_cnf(~alpha))
    a = associate('&',[to_cnf(c) for c in clauses])
    return not dpll_satisfiable(a)

## Exemplo com a base de conhecimento do mundo Wumpus com características temporais

Vamos criar uma base de conhecimento para o mundo `wumpus` mas agora com características temporais.

A base vai ter proposições chamadas `fluentes` cujo valor de verdade depende da variável tempo.

Vamos usar a classe `PropKB` e aplicar as regras do mundo.

In [ ]:
class PropKB(KB):
    """A KB for propositional logic. Inefficient, with no indexing."""

    def __init__(self, sentence=None):
        self.clauses = []
        if sentence:
            self.tell(sentence)

    def tell(self, sentence):
        """Add the sentence's clauses to the KB."""
        self.clauses.extend(conjuncts(to_cnf(sentence)))

    def ask_generator(self, query):
        """Yield the empty substitution {} if KB entails query; else no results."""
        if tt_entails(Expr('&', *self.clauses), query):
            yield {}

    def ask_if_true(self, query):
        """Return True if the KB entails query, else return False."""
        for _ in self.ask_generator(query):
            return True
        return False

    def retract(self, sentence):
        """Remove the sentence's clauses from the KB."""
        for c in conjuncts(to_cnf(sentence)):
            if c in self.clauses:
                self.clauses.remove(c)

In [ ]:
# Expr functions for WumpusKB and HybridWumpusAgent


def facing_east(time):
    return Expr('FacingEast', time)


def facing_west(time):
    return Expr('FacingWest', time)


def facing_north(time):
    return Expr('FacingNorth', time)


def facing_south(time):
    return Expr('FacingSouth', time)


def wumpus(x, y):
    return Expr('W', x, y)


def pit(x, y):
    return Expr('P', x, y)


def breeze(x, y):
    return Expr('B', x, y)


def stench(x, y):
    return Expr('S', x, y)


def wumpus_alive(time):
    return Expr('WumpusAlive', time)


def have_arrow(time):
    return Expr('HaveArrow', time)


def percept_stench(time):
    return Expr('Stench', time)


def percept_breeze(time):
    return Expr('Breeze', time)


def percept_glitter(time):
    return Expr('Glitter', time)


def percept_bump(time):
    return Expr('Bump', time)


def percept_scream(time):
    return Expr('Scream', time)


def move_forward(time):
    return Expr('Forward', time)


def shoot(time):
    return Expr('Shoot', time)


def turn_left(time):
    return Expr('TurnLeft', time)


def turn_right(time):
    return Expr('TurnRight', time)


def ok_to_move(x, y, time):
    return Expr('OK', x, y, time)


def location(x, y, time=None):
    if time is None:
        return Expr('L', x, y)
    else:
        return Expr('L', x, y, time)


# simbolos


def implies(lhs, rhs):
    return Expr('==>', lhs, rhs)


def equiv(lhs, rhs):
    return Expr('<=>', lhs, rhs)


# funções auxiliares


def new_disjunction(sentences):
    t = sentences[0]
    for i in range(1, len(sentences)):
        t |= sentences[i]
    return t


In [ ]:
class WumpusKB(PropKB):
    """
    Criar a base de conhecimento que contem as caracteristicas do "Wumpus físico" e as regras temporais no instante t=0.
    """

    def __init__(self, dimrow):
        super().__init__()
        self.dimrow = dimrow

        ''' o agente sabe que o quarto de inicio não contém nenhum Wumpus nem nenhum poço'''
        self.tell(~wumpus(1, 1))
        self.tell(~pit(1, 1))

        '''Além disso, para cada quadrado, ele sabe que o quadrado é arejado se e só se um quadrado vizinho tiver um poço;
          e um quadrado é malcheiroso se e só se um quadrado vizinho tiver um wumpus.
          Assim, incluímos uma grande coleção de frases do seguinte formato:
            B11 <=> (P12 | P21)
            S11 <=> (W12 | W21)
            '''

        for y in range(1, dimrow + 1):
            for x in range(1, dimrow + 1):

                pits_in = list()
                wumpus_in = list()

                if x > 1:  # West room exists
                    pits_in.append(pit(x - 1, y))
                    wumpus_in.append(wumpus(x - 1, y))

                if y < dimrow:  # North room exists
                    pits_in.append(pit(x, y + 1))
                    wumpus_in.append(wumpus(x, y + 1))

                if x < dimrow:  # East room exists
                    pits_in.append(pit(x + 1, y))
                    wumpus_in.append(wumpus(x + 1, y))

                if y > 1:  # South room exists
                    pits_in.append(pit(x, y - 1))
                    wumpus_in.append(wumpus(x, y - 1))

                self.tell(equiv(breeze(x, y), new_disjunction(pits_in)))
                self.tell(equiv(stench(x, y), new_disjunction(wumpus_in)))

        '''O agente sabe também que existe exactamente um wumpus. Este é expresso em duas partes.
        Primeiro, temos de dizer que existe pelo menos um wumpus:'''
        wumpus_at_least = list()
        for x in range(1, dimrow + 1):
            for y in range(1, dimrow + 1):
                wumpus_at_least.append(wumpus(x, y))

        self.tell(new_disjunction(wumpus_at_least))

        '''Depois temos de dizer que há no máximo um wumpus. Para cada par de locais,
        adicionámos uma frase dizendo que pelo menos um deles deve estar livre de wumpus:'''
        for i in range(1, dimrow + 1):
            for j in range(1, dimrow + 1):
                for u in range(1, dimrow + 1):
                    for v in range(1, dimrow + 1):
                        if i != u or j != v:
                            self.tell(~wumpus(i, j) | ~wumpus(u, v))

        
        '''Regras temporais em t=0 a nossa localização começa por ser L(1,1,0)'''

        self.tell(location(1, 1, 0))

        '''Até aqui tudo bem. Vamos agora considerar as perceções do agente.
          Uma perceção afirma algo apenas sobre o tempo atual. Assim, se o intervalo de tempo
          for 4, então adicionamos Stench(4) à base de conhecimento, em vez de
          simplesmente Stench — evitando qualquer contradição com Stench(3).
          O mesmo acontece com as perceções de brisa, solavanco, brilho e grito.
          Ligamos as perceções de fedor e brisa diretamente às propriedades dos quadrados
          onde são experienciadas da seguinte forma:
          Para qualquer passo de tempo t e qualquer quadrado [x,y]:.
            L(x,y,t) => (Brisa(t) , Bxy)
            L(x,y,t) => (Stencht(0) , Sxy)
                    '''

        for i in range(1, dimrow + 1):
            for j in range(1, dimrow + 1):
                self.tell(implies(location(i, j, 0), equiv(percept_breeze(0), breeze(i, j))))
                self.tell(implies(location(i, j, 0), equiv(percept_stench(0), stench(i, j))))
                if i != 1 or j != 1:
                    self.tell(~location(i, j, 0))


        '''Finalmente o agente sabe os factos que inicializam o jogo'''
        self.tell(wumpus_alive(0))
        self.tell(have_arrow(0))
        self.tell(facing_east(0))
        self.tell(~facing_north(0))
        self.tell(~facing_south(0))
        self.tell(~facing_west(0))

    '''atualiza a base de conhecimento com uma acção que foi realizada no tempo t'''
    def make_action_sentence(self, action, time):
        actions = [move_forward(time), shoot(time), turn_left(time), turn_right(time)]

        for a in actions:
            if action is a:
                self.tell(action)
            else:
                self.tell(~a)

    '''atualiza a base de conhecimento com a perceção que foi sentida no tempo t'''
    def make_percept_sentence(self, percept, time):
        # Glitter, Bump, Stench, Breeze, Scream
        flags = [0, 0, 0, 0, 0]

        # Coisas que foram percecionadas
        if isinstance(percept, Glitter):
            flags[0] = 1
            self.tell(percept_glitter(time))
        elif isinstance(percept, Bump):
            flags[1] = 1
            self.tell(percept_bump(time))
        elif isinstance(percept, Stench):
            flags[2] = 1
            self.tell(percept_stench(time))
        elif isinstance(percept, Breeze):
            flags[3] = 1
            self.tell(percept_breeze(time))
        elif isinstance(percept, Scream):
            flags[4] = 1
            self.tell(percept_scream(time))

        # Coisas que não foram percecionadas
        for i in range(len(flags)):
            if flags[i] == 0:
                if i == 0:
                    self.tell(~percept_glitter(time))
                elif i == 1:
                    self.tell(~percept_bump(time))
                elif i == 2:
                    self.tell(~percept_stench(time))
                elif i == 3:
                    self.tell(~percept_breeze(time))
                elif i == 4:
                    self.tell(~percept_scream(time))

    '''atualiza a base de conhecimento com o estado do agente no tempo t'''
    def add_temporal_sentences(self, time):
        if time == 0:
            return
        t = time - 1

        ''' regras acerca da localização atual'''
        for i in range(1, self.dimrow + 1):
            for j in range(1, self.dimrow + 1):
                self.tell(implies(location(i, j, time), equiv(percept_breeze(time), breeze(i, j))))
                self.tell(implies(location(i, j, time), equiv(percept_stench(time), stench(i, j))))

                '''cria a regra'''
                s = list()

                s.append(
                    equiv(
                        location(i, j, time), location(i, j, time) & ~move_forward(time) | percept_bump(time)))

                if i != 1:
                    s.append(location(i - 1, j, t) & facing_east(t) & move_forward(t))

                if i != self.dimrow:
                    s.append(location(i + 1, j, t) & facing_west(t) & move_forward(t))

                if j != 1:
                    s.append(location(i, j - 1, t) & facing_north(t) & move_forward(t))

                if j != self.dimrow:
                    s.append(location(i, j + 1, t) & facing_south(t) & move_forward(t))

                '''adiciona para o local i,j'''
                self.tell(new_disjunction(s))

                '''regras acerca da possibilidade do movimento'''
                self.tell(
                    equiv(ok_to_move(i, j, time), ~pit(i, j) & ~wumpus(i, j) & wumpus_alive(time))
                )

        '''regras acerca da orientação atual'''

        a = facing_north(t) & turn_right(t)
        b = facing_south(t) & turn_left(t)
        c = facing_east(t) & ~turn_left(t) & ~turn_right(t)
        s = equiv(facing_east(time), a | b | c)
        self.tell(s)

        a = facing_north(t) & turn_left(t)
        b = facing_south(t) & turn_right(t)
        c = facing_west(t) & ~turn_left(t) & ~turn_right(t)
        s = equiv(facing_west(time), a | b | c)
        self.tell(s)

        a = facing_east(t) & turn_left(t)
        b = facing_west(t) & turn_right(t)
        c = facing_north(t) & ~turn_left(t) & ~turn_right(t)
        s = equiv(facing_north(time), a | b | c)
        self.tell(s)

        a = facing_west(t) & turn_left(t)
        b = facing_east(t) & turn_right(t)
        c = facing_south(t) & ~turn_left(t) & ~turn_right(t)
        s = equiv(facing_south(time), a | b | c)
        self.tell(s)

        '''regras acerca da ultima ação'''
        self.tell(equiv(move_forward(t), ~turn_right(t) & ~turn_left(t)))

        '''regras acerca da flecha'''
        self.tell(equiv(have_arrow(time), have_arrow(t) & ~shoot(t)))

        '''regras acerca do wumpus (vivo ou morto)'''
        self.tell(equiv(wumpus_alive(time), wumpus_alive(t) & ~percept_scream(time)))

    
    ''' pergunta à base de conhecimento se é verdadeira uma proposição'''
    def ask_if_true(self, query):
        #return pl_resolution(self, query)
        return pl_resolution_2(self, query)


Vamos construir um agente baseado num modelo

In [ ]:
def ModelBasedReflexAgentProgram(rules, update_state, model):
    """
    This agent takes action based on the percept and state.
    """

    def program(percept):
        program.state = update_state(program.state, program.action, percept, model)
        rule = rule_match(program.state, rules)
        action = rule.action
        return action

    program.state = program.action = None
    return program


def rule_match(state, rules):
    """Find the first rule that matches state."""
    for rule in rules:
        if rule.matches(state):
            return rule

Este é o nosso agente baseado em conhecimento

In [ ]:
class KBAgent(Agent):
    """um agente com base de conhecimentos """
    
    def __init__(self, dimentions):
        self.dimrow = dimentions
        self.kb = WumpusKB(self.dimrow)
        self.t = 0
        self.current_position = [1,1]
        self.current_direction = 0
        super().__init__(self.execute)
        

    def next_square(self,time):
        if self.current_direction == 0 :
            return self.current_position[0],self.current_position[1]+1
        if self.current_direction == 1 :
            return self.current_position[0]+1,self.current_position[1]
        if self.current_direction == 2 :
            return self.current_position[0],self.current_position[1]-1
        if self.current_direction == 3 :
            return self.current_position[0]-1,self.current_position[1]

    def make_action_query(self,time):
        possible_actions = [move_forward(time), shoot(time),turn_left(time), turn_right(time)]
        for a in possible_actions :
            if self.kb.ask_if_true(a):
                print('pode ',a)
                x,y = self.next_square(time)
                if not(a == move_forward(time) and self.kb.ask_if_true(ok_to_move(x,y,time))):
                    return None
                else:
                    return a

        return None
                
        
            
    def make_action_sentence(self, action, time):
        if action == move_forward(time) :
            x,y = self.next_square(time)
            self.current_position = [x,y]
        if action == turn_left(time) :
            self.current_direction = (self.current_direction -1 ) % 4
        if action == turn_right(time) :
            self.current_direction = (self.current_direction + 1) % 4 
        self.kb.make_action_sentence(action, self.t)


   
    def execute(self,percept=None):
        self.kb.make_percept_sentence(percept, self.t)
        self.kb.add_temporal_sentences(self.t)
        action = self.make_action_query(self.t)
        self.make_action_sentence(action, self.t)
        self.t += 1
        return action
    

Vamos então correr o agente

In [ ]:
agent = KBAgent(3)
print('Estas são as suas clausulas na base de conhecimento: \n\n {}'.format(agent.kb.clauses))

In [ ]:
s = agent.kb.clauses
print('Estas são as suas clausulas na base de conhecimento: \n\n {}'.format(s))
print([st for st in s if ('OK' in str(st)) ])
agent.kb.ask_if_true(ok_to_move(1,3,0))

In [ ]:
print('Agente está localizado em {}'.format(agent.current_position))
orientacao = ['Norte','Este','Sul','Oeste']
print('Agente está orientado para {}'.format(orientacao[agent.current_direction]))
print(agent.execute(Stench()))
print('Agente está localizado em {}'.format(agent.current_position))